In [52]:
! pip install nltk
! pip install vaderSentiment
! pip install pytrends
! pip install textblob
! pip install wordcloud
! pip install gensim
! pip install seaborn
! pip install TextBlob
! pip install joblib

  Obtaining dependency information for FuzzyTM>=0.4.0 from https://files.pythonhosted.org/packages/2d/30/074bac7a25866a2807c1005c7852c0139ac22ba837871fc01f16df29b9dc/FuzzyTM-2.0.9-py3-none-any.whl.metadata
  Using cached FuzzyTM-2.0.9-py3-none-any.whl.metadata (7.9 kB)
  Obtaining dependency information for pyfume from https://files.pythonhosted.org/packages/ed/ea/a3b120e251145dcdb10777f2bc5f18b1496fd999d705a178c1b0ad947ce1/pyFUME-0.3.4-py3-none-any.whl.metadata
  Using cached pyFUME-0.3.4-py3-none-any.whl.metadata (9.7 kB)
  Obtaining dependency information for simpful==2.12.0 from https://files.pythonhosted.org/packages/9d/0e/aebc2fb0b0f481994179b2ee2b8e6bbf0894d971594688c018375e7076ea/simpful-2.12.0-py3-none-any.whl.metadata
  Using cached simpful-2.12.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached fst_pso-1.8.1-py3-none-any.whl
  Using cached miniful-0.0.6-py3-none-any.whl
Using cached FuzzyTM-2.0.9-py3-none-any.whl (31 kB)
Using cached pyFUME-0.3.4-py3-none-any.whl (60 kB)
Us

Import Libraries

In [53]:
import nltk
import logging
import numpy as np
import pandas as pd
import seaborn as sns
from textblob import TextBlob
import matplotlib.pyplot as plt
from nltk.corpus import stopwords

import re
import string
import joblib
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [54]:
# Download NLTK resources (if not already downloaded)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('wordnet', quiet=True)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\apoor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\apoor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\apoor\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

Importing Dataset

In [55]:
df = pd.read_csv('Dataset/reviews.csv')
df.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,2818.0,1191.0,3/30/2009,10952,Lam,Daniel is really cool. The place was nice and ...
1,515749.0,1671407.0,7/9/2012,2640670,Gregory,If you want the authentic Amsterdam houseboat ...
2,515749.0,1715674.0,7/15/2012,1032804,Michael,Unique and luxurious to be sure. I couldn't re...
3,2818.0,1771.0,4/24/2009,12798,Alice,Daniel is the most amazing host! His place is ...
4,515749.0,1963378.0,8/12/2012,503786,Brian,My wife and I recently stopped in Amsterdam fo...


Data Preprocessing

In [56]:
df.columns

Index(['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments'], dtype='object')

In [57]:
df.shape

(342904, 6)

In [58]:
df.info

<bound method DataFrame.info of         listing_id            id       date  reviewer_id reviewer_name  \
0           2818.0  1.191000e+03  3/30/2009        10952           Lam   
1         515749.0  1.671407e+06   7/9/2012      2640670       Gregory   
2         515749.0  1.715674e+06  7/15/2012      1032804       Michael   
3           2818.0  1.771000e+03  4/24/2009        12798         Alice   
4         515749.0  1.963378e+06  8/12/2012       503786         Brian   
...            ...           ...        ...          ...           ...   
342899  40606723.0  6.997740e+17  8/23/2022    199478251          Hugo   
342900  40606723.0  6.998450e+17  8/23/2022    168768099         Cerys   
342901  40606723.0  7.004880e+17  8/24/2022    118306945         Anita   
342902  40606723.0  7.004940e+17  8/24/2022    403631092           Tos   
342903  40606723.0  7.005040e+17  8/24/2022    467451204         Laura   

                                                 comments  
0       Daniel is r

In [59]:
df.describe

<bound method NDFrame.describe of         listing_id            id       date  reviewer_id reviewer_name  \
0           2818.0  1.191000e+03  3/30/2009        10952           Lam   
1         515749.0  1.671407e+06   7/9/2012      2640670       Gregory   
2         515749.0  1.715674e+06  7/15/2012      1032804       Michael   
3           2818.0  1.771000e+03  4/24/2009        12798         Alice   
4         515749.0  1.963378e+06  8/12/2012       503786         Brian   
...            ...           ...        ...          ...           ...   
342899  40606723.0  6.997740e+17  8/23/2022    199478251          Hugo   
342900  40606723.0  6.998450e+17  8/23/2022    168768099         Cerys   
342901  40606723.0  7.004880e+17  8/24/2022    118306945         Anita   
342902  40606723.0  7.004940e+17  8/24/2022    403631092           Tos   
342903  40606723.0  7.005040e+17  8/24/2022    467451204         Laura   

                                                 comments  
0       Daniel is

In [60]:
df.isnull().sum()

listing_id        0
id                0
date              0
reviewer_id       0
reviewer_name     0
comments         14
dtype: int64

In [61]:
df.duplicated().sum()

0

In [62]:
# remove the following columns id, date and reviewer_id
df = df.drop(['id', 'reviewer_id', 'listing_id'], axis=1)
df.head()

,date,reviewer_name,comments
0,3/30/2009,Lam,Daniel is really cool. The place was nice and ...
1,7/9/2012,Gregory,If you want the authentic Amsterdam houseboat ...
2,7/15/2012,Michael,Unique and luxurious to be sure. I couldn't re...
3,4/24/2009,Alice,Daniel is the most amazing host! His place is ...
4,8/12/2012,Brian,My wife and I recently stopped in Amsterdam fo...


In [63]:
def remove_urls(text):
    # Input validation
    if not isinstance(text, str):
        logging.error("Input must be a string.")
        raise TypeError("Input must be a string.")

    # Removing URLs from the text
    url_pattern = r'http\S+|www\S+|https\S+'
    cleaned_text = re.sub(url_pattern, '', text, flags=re.MULTILINE)

    return cleaned_text

In [64]:
def remove_mentions_hashtags(text):
    # Input validation
    if not isinstance(text, str):
        logging.error("Input must be a string.")
        raise TypeError("Input must be a string.")

    # Removing mentions and hashtags from the text
    pattern = r'\@\w+|\#\w+'
    cleaned_text = re.sub(pattern, '', text)

    return cleaned_text

In [65]:
def remove_punctuations(text):

    # Input validation
    if not isinstance(text, str):
        logging.error("Input must be a string.")
        raise TypeError("Input must be a string.")

    # Removing punctuations from the text
    translation_table = str.maketrans('', '', string.punctuation)
    cleaned_text = text.translate(translation_table)

    return cleaned_text

In [66]:
# Preload stop words to avoid reloading them every time the function is called
stop_words = set(stopwords.words('english'))


def remove_stopwords(text):
    # Input validation
    if not isinstance(text, str):
        logging.error("Input must be a string.")
        raise TypeError("Input must be a string.")

    # Removing stop words from the text
    filtered_words = [word for word in text.split() if word not in stop_words]
    cleaned_text = ' '.join(filtered_words)

    return cleaned_text

In [67]:
def remove_numbers(text):
    """Remove numbers from a text string"""
    return re.sub(r'\d+', '', text)

In [68]:
def remove_emojis(text):
    """Remove emojis from a text string"""
    emoji_pattern = re.compile(pattern="["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F700-\U0001F77F"  # alchemical symbols
                               u"\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
                               u"\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
                               u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
                               u"\U0001FA00-\U0001FA6F"  # Chess Symbols
                               u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
                               u"\U00002702-\U000027B0"  # Dingbats
                               u"\U000024C2-\U0001F251"
                               "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

In [69]:
def handle_whitespace(text):
    """Handle excessive whitespace"""
    return re.sub(r'\s+', ' ', text).strip()

In [70]:
def lemmitization(text):
    """Lemmitize text"""
    lemmatizer = nltk.stem.WordNetLemmatizer()

In [71]:
def stemminization(text):
    """Stem text"""
    stemmer = nltk.stem.PorterStemmer()
    return ' '.join([stemmer.stem(word) for word in text.split()])

In [72]:
def clean_text(text):
    text = text.lower()
    text = remove_urls(text)
    text = remove_mentions_hashtags(text)
    text = remove_punctuations(text)
    text = remove_stopwords(text)
    text = remove_numbers(text)
    text = remove_emojis(text)
    text = handle_whitespace(text)
    text = lemmitization(text)
    text = stemminization(text)

    # if new functions are added, add them here

    return text

In [73]:
# convert comments column to str type
df['comments'] = df['comments'].astype(str)

In [74]:
# Step 1: Handle missing values
df['comments'] = df['comments'].fillna('')

# Step 2: Safe cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ''
    
    text = text.lower()
    return text

# Step 3: Apply
df['cleaned_comments'] = df['comments'].apply(clean_text)

# Step 4: Check
df.head()

,date,reviewer_name,comments,cleaned_comments
0,3/30/2009,Lam,Daniel is really cool. The place was nice and ...,daniel is really cool. the place was nice and ...
1,7/9/2012,Gregory,If you want the authentic Amsterdam houseboat ...,if you want the authentic amsterdam houseboat ...
2,7/15/2012,Michael,Unique and luxurious to be sure. I couldn't re...,unique and luxurious to be sure. i couldn't re...
3,4/24/2009,Alice,Daniel is the most amazing host! His place is ...,daniel is the most amazing host! his place is ...
4,8/12/2012,Brian,My wife and I recently stopped in Amsterdam fo...,my wife and i recently stopped in amsterdam fo...


In [75]:
df.duplicated().sum()

11

In [76]:
# remove duplicates
df = df.drop_duplicates()
df.duplicated().sum()

0

In [77]:
# Sentiment analysis
df['polarity'] = df['cleaned_comments'].apply(
    lambda x: TextBlob(x).sentiment.polarity)
df['sentiment'] = pd.cut(df['polarity'], bins=3, labels=[
                         'negative', 'neutral', 'positive'])

In [78]:
# print the number of positive, negative and neutral reviews
df['sentiment'].value_counts()

positive    201817
neutral     140162
negative       914
Name: sentiment, dtype: int64

In [79]:
# take 914 positive reviews and 3915 negative reviews and 3915 neutral reviews
df = df.groupby('sentiment').apply(lambda x: x.sample(n=914)).reset_index(
    drop=True).sort_values(by=['sentiment'])
df['sentiment'].value_counts()

negative    914
neutral     914
positive    914
Name: sentiment, dtype: int64

In [88]:
df = pd.read_csv('Dataset/Cleaned_Data/reviews.csv')

# Run again
df['cleaned_comments'] = df['comments'].apply(clean_text)
df['polarity'] = df['cleaned_comments'].apply(lambda x: TextBlob(x).sentiment.polarity)
df['sentiment'] = pd.cut(df['polarity'], bins=3, labels=['negative', 'neutral', 'positive'])

df.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments,cleaned_comments,polarity,sentiment
0,2818.0,1191.0,2009-03-30,10952,Lam,Daniel is really cool. The place was nice and ...,daniel is really cool. the place was nice and ...,0.140741,neutral
1,515749.0,1671407.0,2012-07-09,2640670,Gregory,If you want the authentic Amsterdam houseboat ...,if you want the authentic amsterdam houseboat ...,0.285738,neutral
2,515749.0,1715674.0,2012-07-15,1032804,Michael,Unique and luxurious to be sure. I couldn't re...,unique and luxurious to be sure. i couldn't re...,0.287078,neutral
3,2818.0,1771.0,2009-04-24,12798,Alice,Daniel is the most amazing host! His place is ...,daniel is the most amazing host! his place is ...,0.365278,positive
4,515749.0,1963378.0,2012-08-12,503786,Brian,My wife and I recently stopped in Amsterdam fo...,my wife and i recently stopped in amsterdam fo...,0.172183,neutral


In [89]:
# remove any rows with null values
df = df.dropna()
df.isnull().sum()

listing_id          0
id                  0
date                0
reviewer_id         0
reviewer_name       0
comments            0
cleaned_comments    0
polarity            0
sentiment           0
dtype: int64

In [90]:
df.duplicated().sum()

0

In [91]:
# drop any rows which do not start with a number in the date column
df = df[df['date'].str.contains('^\d', na=False)]

In [92]:
df['date'] = pd.to_datetime(df['date'])

In [93]:
# store the cleaned data
df.to_csv('Dataset/Cleaned_Data/reviews.csv', index=False)